# Sentinel-1 — ALL scenes for Pechea
Exports every matching IW, 10 m pixel spacing, VV+VH scene intersecting the AOI.
Includes partial coverage and different orbits. There is no CHOSEN step.
Each scene produces two PNG previews (VV and VH), not a flood map.
Windows (end exclusive): before 15 August–14 September; during 14–19 September; after 19 September–4 October 2024.


In [ ]:
%pip install -q earthengine-api requests pandas geemap


In [ ]:
"""Sentinel-1 discovery and exports for the Pechea demonstration.

Run in Colab using sentinel1_pechea.ipynb. No credentials are stored here.
PNG files are display previews, not analysis rasters or flood classifications.
"""
import json
from pathlib import Path
import requests
import ee

BBOX = [27.74, 45.58, 27.86, 45.68]  # west, south, east, north
WINDOWS = {
    "before": ("2024-09-10", "2024-09-14"),
    "during": ("2024-09-14", "2024-09-19"),
    "after": ("2024-09-19", "2024-10-04"),
}
COLLECTION = "COPERNICUS/S1_GRD"
EVENT_SLUG = "pechea-2024-09"


def find_scenes(bbox=BBOX, windows=None):
    """Return metadata dictionaries; date ends are exclusive.

    Coverage is an approximate valid VV+VH pixel fraction at 100 m,
    useful for screening, not an accuracy/confidence score.
    """
    windows = WINDOWS if windows is None else windows
    aoi = ee.Geometry.Rectangle(bbox, proj="EPSG:4326", geodesic=False)
    base = (ee.ImageCollection(COLLECTION).filterBounds(aoi)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.eq("resolution_meters", 10))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))

    def record(image):
        valid = image.select(["VV", "VH"]).mask().reduce(ee.Reducer.min())
        coverage = valid.unmask(0, False).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=aoi, scale=100,
            maxPixels=1000000).values().get(0)
        return image.set("record", ee.Dictionary({
            "asset_id": ee.String(COLLECTION).cat("/").cat(ee.String(image.get("system:index"))),
            "acquired_at": image.date().format("yyyy-MM-dd'T'HH:mm:ss'Z'", "UTC"),
            "pass": image.get("orbitProperties_pass"),
            "relative_orbit": image.get("relativeOrbitNumber_start"),
            "platform": image.get("platform_number"),
            "coverage_fraction_approx": coverage,
            "pixel_spacing_m": 10,
            "bands": ["VV", "VH"],
            "units": "dB",
        }))

    records = []
    for period, (start, end) in windows.items():
        items = (base.filterDate(start, end).sort("system:time_start")
                 .map(record).aggregate_array("record").getInfo())
        records.extend(dict(item, period=period) for item in items)
    return records


def download(url, path, signature):
    """Validate the response before replacing a local export."""
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    if not response.content.startswith(signature):
        raise ValueError(f"Unexpected file format for {path.name}")
    path.write_bytes(response.content)


def export_scenes(records, output_dir="export/sentinel1_all",
                  bbox=BBOX, export_geotiff=False):
    """Download stable PNGs, metadata and optional quantitative GeoTIFFs.

    PNGs use EPSG:3857 to align with the app's Leaflet map projection.
    All scenes use identical display limits per band. No cloud filter is needed.
    """
    root = Path(output_dir)
    folder = root / "data" / "pechea"
    folder.mkdir(parents=True, exist_ok=True)
    aoi = ee.Geometry.Rectangle(bbox, proj="EPSG:4326", geodesic=False)
    manifest = []

    for item in records:
        image = ee.Image(item["asset_id"]).select(["VV", "VH"]).clip(aoi)
        scene_id = item["asset_id"].rsplit("/", 1)[-1]
        stem = f"s1_{item['period']}_{scene_id}"
        for band, limits in {"VV": (-25, 0), "VH": (-30, -5)}.items():
            filename = f"{stem}_{band.lower()}.png"
            url = image.select(band).getThumbURL({
                "min": limits[0], "max": limits[1],
                "palette": ["000000", "ffffff"],
                "region": aoi, "crs": "EPSG:3857",
                "dimensions": 1400, "format": "png",
            })
            download(url, folder / filename, b"\x89PNG\r\n\x1a\n")
            manifest.append({
                "id": f"{stem}_{band.lower()}",
                "source": "Sentinel-1",
                "period": item["period"],
                "type": "image",
                "url": f"/data/pechea/{filename}",
                "bounds": list(bbox),
                "acquired_at_utc": item["acquired_at"],
                "status": "available",
            })
            print("Downloaded", filename)

        if export_geotiff:
            # Small AOI: below the EE direct download size limit.
            url = image.toFloat().getDownloadURL({
                "region": aoi, "scale": 10, "crs": "EPSG:32635",
                "format": "ZIPPED_GEO_TIFF", "filePerBand": False,
            })
            download(url, root / f"{stem}_db_geotiff.zip", b"PK")

    # Keep provenance separate so the frontend contract remains exact.
    (root / "candidates.json").write_text(json.dumps(records, indent=2))
    (root / "layers_sentinel1.json").write_text(json.dumps(manifest, indent=2))
    print("Saved metadata to", root)
    return manifest


def get_sentinel1_layers(output_dir="export/sentinel1_all", bbox=BBOX,
                        export_geotiff=False):
    """Return two image records per matching scene, across all three windows.

    Includes every matching IW 10 m VV+VH scene intersecting the AOI,
    regardless of pass/orbit/platform or partial coverage. No selection,
    mosaicking, date averaging or flood classification is performed.
    Empty searches return []; download errors raise rather than marking
    missing files as available. Orbit metadata stays in candidates.json.
    """
    records = find_scenes(bbox=bbox)
    return export_scenes(records, output_dir, bbox, export_geotiff)


In [ ]:
PROJECT_ID = "dotted-signer-423510-g2"
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)


In [ ]:
import pandas as pd
from IPython.display import display
records = find_scenes()
if records:
    display(pd.DataFrame(records))
for period in WINDOWS:
    count = sum(r["period"] == period for r in records)
    print(f"{period}: {count} scenes -> {2 * count} PNG previews")


,acquired_at,asset_id,bands,coverage_fraction_approx,pass,pixel_spacing_m,platform,relative_orbit,units,period
0,2024-09-10T16:01:15Z,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240910T16...,"[VV, VH]",1,ASCENDING,10,A,58,dB,before
1,2024-09-10T16:01:15Z,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240910T16...,"[VV, VH]",1,ASCENDING,10,A,58,dB,before
2,2024-09-14T04:21:50Z,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240914T04...,"[VV, VH]",1,DESCENDING,10,A,109,dB,during
3,2024-09-21T04:13:41Z,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240921T04...,"[VV, VH]",1,DESCENDING,10,A,36,dB,after
4,2024-09-22T16:01:16Z,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240922T16...,"[VV, VH]",1,ASCENDING,10,A,58,dB,after
5,2024-09-26T04:21:50Z,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240926T04...,"[VV, VH]",1,DESCENDING,10,A,109,dB,after
6,2024-10-03T04:13:42Z,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20241003T04...,"[VV, VH]",1,DESCENDING,10,A,36,dB,after


before: 2 scenes -> 4 PNG previews
during: 1 scenes -> 2 PNG previews
after: 4 scenes -> 8 PNG previews


In [ ]:
for record in records:
    if not record["asset_id"].startswith(COLLECTION + "/"):
        record["asset_id"] = COLLECTION + "/" + record["asset_id"]

print(records[0]["asset_id"])

COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240910T160115_20240910T160140_055605_06C9B1_2B02


In [ ]:
import geemap
m = geemap.Map()
aoi = ee.Geometry.Rectangle(BBOX, proj="EPSG:4326", geodesic=False)
m.centerObject(aoi, 12)
for index, item in enumerate(records):
    image = ee.Image(item["asset_id"]).clip(aoi)
    scene_id = item["asset_id"].rsplit("/", 1)[-1]
    for band, limits in {"VV": (-25, 0), "VH": (-30, -5)}.items():
        label = (f"{item['period']} | {band} | {item['acquired_at']} | "
                 f"{item['pass']} orbit {item['relative_orbit']} | {scene_id}")
        m.addLayer(image.select(band), {"min": limits[0], "max": limits[1]},
                   label, index == 0 and band == "VV")
m


Map(center=[45.62999036944863, 27.799999999998416], controls=(WidgetControl(options=['position', 'transparent_…

All scenes are included. Toggle layers to inspect them. Different orbits have different viewing geometries: retain all observations for browsing, but group comparable orbits before computing changes. A dark area is not automatically flooding. PNG previews are not analysis rasters.


In [ ]:
layers = export_scenes(records, export_geotiff=False)
display(pd.DataFrame(layers))
# Optional: export_geotiff=True also downloads quantitative VV/VH dB rasters.


Downloaded s1_before_S1A_IW_GRDH_1SDV_20240910T160115_20240910T160140_055605_06C9B1_2B02_vv.png
Downloaded s1_before_S1A_IW_GRDH_1SDV_20240910T160115_20240910T160140_055605_06C9B1_2B02_vh.png
Downloaded s1_before_S1A_IW_GRDH_1SDV_20240910T160115_20240910T160140_055605_06C9B1_7585_vv.png
Downloaded s1_before_S1A_IW_GRDH_1SDV_20240910T160115_20240910T160140_055605_06C9B1_7585_vh.png
Downloaded s1_during_S1A_IW_GRDH_1SDV_20240914T042150_20240914T042215_055656_06CBAD_ACFC_vv.png
Downloaded s1_during_S1A_IW_GRDH_1SDV_20240914T042150_20240914T042215_055656_06CBAD_ACFC_vh.png
Downloaded s1_after_S1A_IW_GRDH_1SDV_20240921T041341_20240921T041406_055758_06CFB4_05EA_vv.png
Downloaded s1_after_S1A_IW_GRDH_1SDV_20240921T041341_20240921T041406_055758_06CFB4_05EA_vh.png
Downloaded s1_after_S1A_IW_GRDH_1SDV_20240922T160116_20240922T160141_055780_06D090_2DA7_vv.png
Downloaded s1_after_S1A_IW_GRDH_1SDV_20240922T160116_20240922T160141_055780_06D090_2DA7_vh.png
Downloaded s1_after_S1A_IW_GRDH_1SDV_2024092

,id,source,period,type,url,bounds,acquired_at_utc,status
0,s1_before_S1A_IW_GRDH_1SDV_20240910T160115_202...,Sentinel-1,before,image,/data/pechea/s1_before_S1A_IW_GRDH_1SDV_202409...,"[27.74, 45.58, 27.86, 45.68]",2024-09-10T16:01:15Z,available
1,s1_before_S1A_IW_GRDH_1SDV_20240910T160115_202...,Sentinel-1,before,image,/data/pechea/s1_before_S1A_IW_GRDH_1SDV_202409...,"[27.74, 45.58, 27.86, 45.68]",2024-09-10T16:01:15Z,available
2,s1_before_S1A_IW_GRDH_1SDV_20240910T160115_202...,Sentinel-1,before,image,/data/pechea/s1_before_S1A_IW_GRDH_1SDV_202409...,"[27.74, 45.58, 27.86, 45.68]",2024-09-10T16:01:15Z,available
3,s1_before_S1A_IW_GRDH_1SDV_20240910T160115_202...,Sentinel-1,before,image,/data/pechea/s1_before_S1A_IW_GRDH_1SDV_202409...,"[27.74, 45.58, 27.86, 45.68]",2024-09-10T16:01:15Z,available
4,s1_during_S1A_IW_GRDH_1SDV_20240914T042150_202...,Sentinel-1,during,image,/data/pechea/s1_during_S1A_IW_GRDH_1SDV_202409...,"[27.74, 45.58, 27.86, 45.68]",2024-09-14T04:21:50Z,available
5,s1_during_S1A_IW_GRDH_1SDV_20240914T042150_202...,Sentinel-1,during,image,/data/pechea/s1_during_S1A_IW_GRDH_1SDV_202409...,"[27.74, 45.58, 27.86, 45.68]",2024-09-14T04:21:50Z,available
6,s1_after_S1A_IW_GRDH_1SDV_20240921T041341_2024...,Sentinel-1,after,image,/data/pechea/s1_after_S1A_IW_GRDH_1SDV_2024092...,"[27.74, 45.58, 27.86, 45.68]",2024-09-21T04:13:41Z,available
7,s1_after_S1A_IW_GRDH_1SDV_20240921T041341_2024...,Sentinel-1,after,image,/data/pechea/s1_after_S1A_IW_GRDH_1SDV_2024092...,"[27.74, 45.58, 27.86, 45.68]",2024-09-21T04:13:41Z,available
8,s1_after_S1A_IW_GRDH_1SDV_20240922T160116_2024...,Sentinel-1,after,image,/data/pechea/s1_after_S1A_IW_GRDH_1SDV_2024092...,"[27.74, 45.58, 27.86, 45.68]",2024-09-22T16:01:16Z,available
9,s1_after_S1A_IW_GRDH_1SDV_20240922T160116_2024...,Sentinel-1,after,image,/data/pechea/s1_after_S1A_IW_GRDH_1SDV_2024092...,"[27.74, 45.58, 27.86, 45.68]",2024-09-22T16:01:16Z,available


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("sentinel1_all_scenes_exports", "zip", "export/sentinel1_all")
files.download(archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

`layers_sentinel1.json` contains the exact eight team fields. IDs and filenames include the full scene identifier to prevent overwriting scenes acquired on the same day.
Copy `data/pechea/*.png` into `frontend/public/data/pechea/` so the agreed URLs resolve. Keep `candidates.json` for orbit and source metadata. Empty periods have no entries; an empty search returns an empty list.
